# Introduction: Real-Time Communication using WebSockets
In this module, we'll learn how WebSockets enable persistent, bidirectional communication between clients and servers.

> __Learning Objectives__
>
> By the end of this module, you will be able to:
> * __Describe the WebSocket architecture:__ Explain the HTTP-based handshake process that upgrades a connection from the traditional request-response model to a persistent frame-based channel.
> * __Parse and construct WebSocket frames:__ Understand the header fields (FIN, opcode, mask, length, etc.) and how text vs. binary payloads are handled.

As applications demand ever-lower latency and instantaneous updates, WebSockets provide a single, long-lived TCP socket over which both the client and server can push messages at any time, making them ideal for chat, live dashboards, multiplayer games, and real-time trading platforms.

To appreciate why WebSockets have become essential for modern real-time applications, let's first understand what makes them different from traditional HTTP. Let's get started!
___

## Why a WebSocket?
WebSockets bridge the gap between the simple HTTP request-response model and the needs of modern real-time applications. Three key advantages explain their importance:

> __Persistence__
>
> Your client and server maintain a constant, two-way conversation, allowing messages to flow instantly over a persistent connection. WebSockets eliminate the need for repeated HTTP requests and responses (called polling), which can introduce latency and overhead, and can miss data updates.

Beyond persistence, WebSockets also optimize performance:

> __Lower latency & overhead__
>
> After an initial HTTP-based handshake, a single TCP connection is kept open. There is no per-message HTTP header exchange, so payloads are smaller and round-trip delays drop. WebSockets let the client and server push updates onto the connection the moment they happen (which is unlike traditional HTTP, where the client must request data from the server).

Finally, WebSockets enable true bidirectional communication:

> __Full-duplex channel__
>
> Both client and server can send independently at any time. This makes WebSockets ideal for chat apps, live dashboards, multiplayer games, IoT telemetry, and collaborative editing, among other applications.

Understanding these advantages is important, but to use WebSockets effectively, we need to understand how they establish connections. Let's examine the handshake process that makes this real-time communication possible.
___

## Handshake
Every client, e.g., a browser, a mobile app, or an IoT device, must first perform a quick, HTTP-based handshake. Both sides of the conversation must agree to switch from the traditional request-response model to a long-lived, bidirectional TCP connection. The handshake follows four steps:

> __1. Client asks to upgrade__
>
> Your application sends a standard HTTP `GET` request to the server, but adds special headers indicating "I'd like to switch to WebSockets," ensuring the underlying TCP connection between the client and server stays open.

The server then processes this request:

> __2. Server verifies and confirms__
>
> The server checks a nonce to prevent spoofing, then replies with `101 Switching Protocols` and a matching token to say "OK, let's do this."

With agreement from both sides, the protocol transition occurs:

> __3. Protocol switch__
>
> Once confirmed, both sides stop using the request-response model and exchanging HTTP messages and instead begin exchanging compact WebSocket frames over the same TCP connection.

Finally, the connection is ready for real-time communication:

> __4. Persistent, two-way channel__
>
> The TCP socket remains open indefinitely (until closed), allowing your client app and server to send messages to each other at any time with minimal framing overhead. On the server side, new data can be pushed to the client as soon as it's available, without waiting for a request.

Two technical concepts in this handshake process deserve closer attention: nonces and the 101 status code.

### What is a nonce?
A nonce (short for "number used once") is a random, single-use value that helps ensure each handshake or transaction is unique and cannot be replayed. Think of it as a temporary password that is valid only for the current session.

> __Uniqueness__
>
> Every time you initiate a WebSocket (or any secure) handshake, you generate a fresh nonce so the server can tell this is a brand-new request. Every nonce is unique to that specific handshake.

This uniqueness provides an important security property:

> __Proof against replay__
>
> Because the server signs or echoes back the nonce (after hashing), an attacker cannot capture an old handshake and replay it to trick your server. This ensures that even if someone intercepts the handshake, they cannot reuse it later, e.g., to hijack a session. Thus, using a nonce adds minimal overhead but has a significant security benefit.

### What is 101 Switching Protocols?
The HTTP `101 Switching Protocols` code is an HTTP/1.1 status code the server sends back to say, "Okay, I accept your request to switch from HTTP to the new protocol (e.g., WebSocket)."

> __Status 1xx (Informational)__
>
> The 1xx set of status codes does not carry a page or any data; they just let the client know that the server is processing the request and has accepted the protocol switch.

This status code has a specific purpose:

> __"Switching Protocols"__
>
> Echoes the client's desire to upgrade, confirming both sides will stop speaking HTTP and start using WebSocket frames.

Importantly, the switch happens on the existing connection:

> __Stays on the same TCP socket__
>
> No new connection is opened; all communication will happen on the existing socket, now running the agreed-upon protocol.

Once the handshake succeeds, the client and server can now exchange messages using WebSocket frames. These frames are significantly lighter than traditional HTTP requests, enabling faster and more efficient communication. Let's examine how these frames work.
___

## Frames
Once the handshake completes, client and server exchange data in WebSocket frames, which are self-contained packets that carry both metadata (stored in a compact header) and your actual message (the payload). Below is the header format for each frame:

| Field            | Size             | Description                                                                                                                                                          |
| ---------------- | ---------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **FIN + RSV1–3** | 4 bits           | **FIN:** set to 1 on the final frame of a message. <br> **RSV1–3:** reserved for negotiated extensions (e.g. compression). Usually 0.                                |
| **Opcode**       | 4 bits           | Type of frame:<br>• 0 = continuation frame<br>• 1 = text data<br>• 2 = binary data<br>• 8 = connection close<br>• 9 = ping<br>• 10 = pong                            |
| **Mask bit**     | 1 bit            | Client→server frames **must** set this to 1 (so the server knows payload is masked). Server→client frames **must** set to 0.                                         |
| **Payload len**  | 7 bits (+ ext)   | If ≤ 125, that value is the payload length in bytes. <br> If 126, the next 2 bytes (16-bit) give the length. <br> If 127, the next 8 bytes (64-bit) give the length. |
| **Extended len** | 0, 2, or 8 bytes | Present only when the 7-bit length is 126 or 127, to hold the full payload length.                                                                                   |
| **Masking key**  | 0 or 4 bytes     | A 4-byte random key used to unmask the payload. Present **only** when Mask bit = 1.                                                                                  |
| **Payload data** | (variable)       | The actual application data. If masked, each byte is XOR'd with the masking key.                                                                                     |

This frame structure might seem complex, but you rarely need to work with it directly:

> __In practice__
>
> Your client application library will handle framing for you. Your job is just to send or receive messages. But under the hood, every send and receive corresponds to one or more of these frames, providing the reliability, control, and efficiency that power real-time WebSocket apps.

Understanding how the library handles this complexity reveals an important design principle:

> __Why frames?__
>
> They allow WebSockets to be lightweight and efficient. Instead of sending large HTTP headers with every message, WebSocket frames have a compact header that carries just enough metadata to route and process the message. This reduces overhead and improves performance, especially for high-frequency updates.

In Julia, you can use [the `HTTP.jl` package](https://github.com/JuliaWeb/HTTP.jl) to work with WebSockets. The package provides a simple interface for creating WebSocket clients and servers, handling the framing and message parsing for you.

Coincidentally, the `HTTP.jl` package is the same package that we used when building our RESTful API in the previous module. Always buy, never build!
___

## Summary
This module introduced WebSockets as a protocol for persistent, bidirectional communication between clients and servers, enabling real-time applications.

> __Key Takeaways:__
>
> * **WebSockets provide persistent connections:** Unlike traditional HTTP request-response cycles, WebSockets maintain a single long-lived TCP connection that allows both client and server to push messages at any time, reducing latency and overhead.
> * **Handshake upgrades HTTP to WebSocket protocol:** The connection begins with an HTTP handshake that includes a nonce for security and a 101 status code for protocol switching, after which both sides exchange compact frames instead of HTTP messages.
> * **Frames optimize real-time data transfer:** WebSocket frames carry minimal metadata in a compact header, making them significantly more efficient than HTTP requests for high-frequency updates in applications like chat, dashboards, and games.

WebSockets enable the low-latency, bidirectional communication required for modern real-time applications, bridging the gap between simple HTTP and the demands of instant data exchange.

___